In [2]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import JinaEmbeddings

C:\Users\parth\AppData\Local\Temp\ipykernel_3544\3588720268.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
c:\Users\parth\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
load_dotenv()

True

In [4]:
groq_api_key =os.getenv("GROQ_API_KEY")
jina_api_key =os.getenv("JINA_API_KEY")

print("env loaded")


env loaded


LOADING OUR DATA

In [5]:
DATA_FILE_PATH =os.path.join("data","hr_policy.txt")

DATA INGESTION

loader = TextLoader(DATA_FILE_PATH,encoding = "")

In [6]:
loader = TextLoader(DATA_FILE_PATH,encoding = "utf-8")

documents = loader.load()

print(documents[0].page_content)

COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)

1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

2. WORK FROM HOME POLICY
Employees may work from home up to 2 days per week, subject to manager approval.
Fully remote work arrangements require written approval from the department head.
Employees working from home must be reachable during core hours: 10 AM to 4 PM.

3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of e

In [7]:
print(documents[0].metadata)

{'source': 'data\\hr_policy.txt'}


SPLITTING OUR DATA

In [ ]:
text_splitter =RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(documents)
print(len(chunks))

9


In [9]:
print(chunks[0])

page_content='COMPANY HR POLICY HANDBOOK
Acme Corp - Employee Handbook (Demo Document)' metadata={'source': 'data\\hr_policy.txt'}


In [10]:
print(chunks[8])

page_content='8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.' metadata={'source': 'data\\hr_policy.txt'}


EMBEDED OUR DATA

In [11]:
emdedding_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")
print("emdedding model ready the name is " ,emdedding_model.model_name)

emdedding model ready the name is  jina-embeddings-v2-base-en


STORE THE DATA IN VECTOR DATABASE

In [12]:
! pip install faiss-cpu


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks,emdedding_model)

print("success")

success


In [14]:
test_query= "How many sick leaves employees get"

#similarity search

top_matches =  vector_store.similarity_search(test_query,k=2)
print(f"Query: {test_query}")
for i,match in enumerate(top_matches,start=1):
    print(f"----Match {i} ---")
    print(match.page_content)
    print()


Query: How many sick leaves employees get
----Match 1 ---
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.

----Match 2 ---
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.



TOOL

In [15]:
retriever = vector_store.as_retriever(search_kwargs = {"k":3})

def search_hr_policy(question:str)-> str:
    """
    Search the HR policy document for information about leave, work from Home probation,
    notice period, reimbursement, code of conduct, holidays, or exit process. 
    """
    matching_chunks = retriever.invoke(question)
    return "".join(chunk.page_content for chunk in matching_chunks)

DATA RETRIVAL

LLM

In [16]:
! pip install langchain-groq



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [17]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b",
    temperature = 0
)

llm.model_name

'openai/gpt-oss-120b'

In [18]:
response = llm.invoke("Hey is learning Rag hard? answer this question, in one line")

In [19]:
print(response.content)

Learning RAG can be challenging at first, but with good resources and practice it’s definitely manageable.


AI AGENT

In [20]:
from langchain.agents import create_agent


In [26]:
hr_assistant = create_agent(
    model = llm,
    tools = [search_hr_policy],
    system_prompt= """ 
    You are a friendly HR assistant.
    Always use the search_hr_policy tool to look up.
    facts before answering.
    If the answer isnot in the search results, say you dont known "
    instead of guessing."
    """
)

In [22]:
def ask_question(question: str) -> str:
    """ Send a question to the RAG agent and print a nicely formatted answer."""
    print("-"*40)
    print("QUESTION:", question)
    print("-"*40)

    response = hr_assistant.invoke({"messages":[{"role":"user","content":question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("-"*40)
    print()

    return answer

In [27]:
response = hr_assistant.invoke(
    {
      "messages":[
        {
          "role": "user",
          "content":" tell me about leave polies how to apply for leave"
        }
      ]
    }
  )

In [28]:
response

{'messages': [HumanMessage(content=' tell me about leave polies how to apply for leave', additional_kwargs={}, response_metadata={}, id='30a807a0-33a0-455f-9b64-6a482a190244'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to follow developer instructions: always use search_hr_policy tool to look up facts before answering. So we must call search_hr_policy with a question about leave policies and how to apply. Then based on results, answer. If not found, say "I don\'t know". Let\'s call the tool.', 'tool_calls': [{'id': 'fc_00b2548e-26bb-4a0f-9eb3-275494a81179', 'function': {'arguments': '{"question":"What are the leave policies and how to apply for leave?"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 205, 'total_tokens': 304, 'completion_time': 0.214330064, 'completion_tokens_details': {'reasoning_tokens': 61}, 'prompt_time': 0.009192085, 'prompt_tokens_details': None, 'que

In [30]:
response["messages"][-1]

AIMessage(content='**Leave Policies Overview**\n\n| Type of Leave | Entitlement | Key Rules |\n|---------------|------------|-----------|\n| **Annual (Paid) Leave** | 20 days per calendar year (full‑time) | • Must be requested **at least 5 working days** before the intended start date.<br>• Requests are submitted through the **HR portal**.<br>• Up to **5 unused days** can be carried forward to the next year; any excess lapses. |\n| **Sick Leave** | 10 paid days per year | • Separate from annual leave.<br>• A **medical certificate** is required if the sick leave exceeds **2 consecutive days**. |\n| **Public Holiday** | 12 days (as per the official holiday calendar) | • If you work on a public holiday, you are eligible for **compensatory leave**. |\n\n**How to Apply for Leave**\n\n1. **Log into the HR Portal**  \n   - Use your employee credentials to access the portal (usually at `hr.company.com` or via the intranet).\n\n2. **Navigate to the “Leave Request” Section**  \n   - Click **“New